# 03 – Data Integration

## Objective
This notebook combines the cleaned bicycle traffic data
and weather data from Zurich for the years 2023–2025.

## Steps
1. Load cleaned datasets
2. Inspect data structures
3. Validate timestamps
4. Merge bicycle and weather data
5. Evaluate missing values and unmatched records
6. Save the integrated dataset

In [1]:
import pandas as pd
from pathlib import Path

# Locate project root
cwd = Path.cwd()

project_root = next(
    p for p in [cwd, *cwd.parents]
    if (p / "data").is_dir()
    and (p / "notebooks").is_dir()
)

processed_dir = project_root / "data" / "processed"

# List available processed files
for file in processed_dir.glob("*.csv"):
    print(file.name)

velo_hourly_2023_2025.csv
weather_zurich_hourly_2023_2025_cleaned.csv


In [2]:
# Load cleaned bicycle traffic data
velo_path = processed_dir / "velo_hourly_2023_2025.csv"

velo = pd.read_csv(velo_path)

# Show sample and data structure
display(velo.head())

print("Rows:", len(velo))
print("Columns:", velo.columns.tolist())

# Check missing values
display(velo.isna().sum())

,FK_STANDORT,stunde,velo_in,velo_out,anzahl_intervalle,messungen_in,messungen_out,velo_total
0,60,01/01/2023 00:00,0,1.0,4,4,4,1
1,60,01/01/2023 01:00,0,2.0,4,4,4,2
2,60,01/01/2023 02:00,0,3.0,4,4,4,3
3,60,01/01/2023 03:00,2,2.0,4,4,4,4
4,60,01/01/2023 04:00,0,0.0,4,4,4,0


Rows: 587954
Columns: ['FK_STANDORT', 'stunde', 'velo_in', 'velo_out', 'anzahl_intervalle', 'messungen_in', 'messungen_out', 'velo_total']


FK_STANDORT              0
stunde                   0
velo_in                  0
velo_out             68774
anzahl_intervalle        0
messungen_in             0
messungen_out            0
velo_total               0
dtype: int64

## Weather Data Inspection

Before integrating both datasets, the cleaned weather
data is inspected to identify its structure, columns,
timestamps, and missing values.

In [3]:
# Search for cleaned weather datasets
weather_files = [
    file for file in processed_dir.glob("*.csv")
    if any(
        keyword in file.name.lower()
        for keyword in ["weather", "wetter", "meteo"]
    )
]

print("Available weather datasets:")

for file in weather_files:
    print(file.name)

Available weather datasets:
weather_zurich_hourly_2023_2025_cleaned.csv


In [4]:
# Select the weather dataset
if len(weather_files) != 1:
    raise ValueError(
        "Please select exactly one cleaned weather CSV file."
    )

weather_path = weather_files[0]

# Load weather data
weather = pd.read_csv(weather_path)

# Display sample records
display(weather.head())

# Inspect dataset structure
print("Rows:", len(weather))
print("Columns:", weather.columns.tolist())

# Check data types
weather.info()

# Check missing values
display(weather.isna().sum())

,time,temperature_2m,precipitation,relative_humidity_2m,wind_speed_10m,cloud_cover,is_rainy,temperature_category
0,2023-01-01 00:00:00+00:00,8.1,0.0,84,5.1,74,False,0-10°C
1,2023-01-01 01:00:00+00:00,8.2,0.0,83,4.6,48,False,0-10°C
2,2023-01-01 02:00:00+00:00,8.4,0.0,85,3.6,48,False,0-10°C
3,2023-01-01 03:00:00+00:00,8.0,0.0,86,2.9,61,False,0-10°C
4,2023-01-01 04:00:00+00:00,8.0,0.0,86,3.2,41,False,0-10°C


Rows: 26304
Columns: ['time', 'temperature_2m', 'precipitation', 'relative_humidity_2m', 'wind_speed_10m', 'cloud_cover', 'is_rainy', 'temperature_category']
<class 'pandas.DataFrame'>
RangeIndex: 26304 entries, 0 to 26303
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   time                  26304 non-null  str    
 1   temperature_2m        26304 non-null  float64
 2   precipitation         26304 non-null  float64
 3   relative_humidity_2m  26304 non-null  int64  
 4   wind_speed_10m        26304 non-null  float64
 5   cloud_cover           26304 non-null  int64  
 6   is_rainy              26304 non-null  bool   
 7   temperature_category  26304 non-null  str    
dtypes: bool(1), float64(3), int64(2), str(2)
memory usage: 1.4 MB


time                    0
temperature_2m          0
precipitation           0
relative_humidity_2m    0
wind_speed_10m          0
cloud_cover             0
is_rainy                0
temperature_category    0
dtype: int64

## Timestamp Harmonization

The bicycle traffic data uses local Zurich time,
while the weather data uses UTC.

Weather timestamps are converted to the Europe/Zurich
timezone before merging both datasets.

Special attention is given to daylight saving time.

In [5]:
# Create a copy of the weather dataset
weather_local = weather.copy()

# Convert UTC timestamps to Zurich local time
weather_local["stunde"] = (
    pd.to_datetime(weather_local["time"], utc=True)
    .dt.tz_convert("Europe/Zurich")
    .dt.tz_localize(None)
)

# Inspect converted timestamps
display(weather_local[["time", "stunde"]].head())

# Check duplicate local timestamps
duplicates = weather_local["stunde"].duplicated().sum()

print("Duplicate local timestamps:", duplicates)
print("First local timestamp:", weather_local["stunde"].min())
print("Last local timestamp:", weather_local["stunde"].max())

,time,stunde
0,2023-01-01 00:00:00+00:00,2023-01-01 01:00:00
1,2023-01-01 01:00:00+00:00,2023-01-01 02:00:00
2,2023-01-01 02:00:00+00:00,2023-01-01 03:00:00
3,2023-01-01 03:00:00+00:00,2023-01-01 04:00:00
4,2023-01-01 04:00:00+00:00,2023-01-01 05:00:00


Duplicate local timestamps: 3
First local timestamp: 2023-01-01 01:00:00
Last local timestamp: 2026-01-01 00:00:00


In [6]:
# Aggregate weather measurements by local hour
weather_hourly = (
    weather_local.groupby("stunde", as_index=False)
    .agg(
        temperature_2m=("temperature_2m", "mean"),
        precipitation=("precipitation", "sum"),
        relative_humidity_2m=("relative_humidity_2m", "mean"),
        wind_speed_10m=("wind_speed_10m", "mean"),
        cloud_cover=("cloud_cover", "mean"),
        is_rainy=("is_rainy", "max")
    )
)

# Validate the aggregated dataset
print("Original weather rows:", len(weather_local))
print("Unique local hours:", len(weather_hourly))

print(
    "Duplicate timestamps:",
    weather_hourly["stunde"].duplicated().sum()
)

display(weather_hourly.head())

Original weather rows: 26304
Unique local hours: 26301
Duplicate timestamps: 0


,stunde,temperature_2m,precipitation,relative_humidity_2m,wind_speed_10m,cloud_cover,is_rainy
0,2023-01-01 01:00:00,8.1,0.0,84.0,5.1,74.0,False
1,2023-01-01 02:00:00,8.2,0.0,83.0,4.6,48.0,False
2,2023-01-01 03:00:00,8.4,0.0,85.0,3.6,48.0,False
3,2023-01-01 04:00:00,8.0,0.0,86.0,2.9,61.0,False
4,2023-01-01 05:00:00,8.0,0.0,86.0,3.2,41.0,False


In [7]:
# Convert bicycle timestamps to datetime
velo["stunde"] = pd.to_datetime(
    velo["stunde"],
    errors="coerce"
)

# Inspect bicycle data
display(velo.head())

print("Bicycle rows:", len(velo))
print("First timestamp:", velo["stunde"].min())
print("Last timestamp:", velo["stunde"].max())

# Check duplicate station-hour combinations
duplicate_velo = velo.duplicated(
    subset=["FK_STANDORT", "stunde"]
).sum()

print("Duplicate station-hour combinations:", duplicate_velo)

# Check invalid timestamps
print("Invalid timestamps:", velo["stunde"].isna().sum())

,FK_STANDORT,stunde,velo_in,velo_out,anzahl_intervalle,messungen_in,messungen_out,velo_total
0,60,2023-01-01 00:00:00,0,1.0,4,4,4,1
1,60,2023-01-01 01:00:00,0,2.0,4,4,4,2
2,60,2023-01-01 02:00:00,0,3.0,4,4,4,3
3,60,2023-01-01 03:00:00,2,2.0,4,4,4,4
4,60,2023-01-01 04:00:00,0,0.0,4,4,4,0


Bicycle rows: 587954
First timestamp: 2023-01-01 00:00:00
Last timestamp: 2025-12-12 23:00:00
Duplicate station-hour combinations: 355943
Invalid timestamps: 355976


In [10]:
# Check missing bicycle timestamps
missing_timestamps = velo["stunde"].isna().sum()

print("Total bicycle rows:", len(velo))
print("Missing timestamps:", missing_timestamps)

# Display affected rows
display(
    velo[velo["stunde"].isna()].head(10)
)

Total bicycle rows: 587954
Missing timestamps: 355976


,FK_STANDORT,stunde,velo_in,velo_out,anzahl_intervalle,messungen_in,messungen_out,velo_total
288,60,NaT,1,4.0,4,4,4,5
289,60,NaT,3,2.0,4,4,4,5
290,60,NaT,2,1.0,4,4,4,3
291,60,NaT,1,0.0,4,4,4,1
292,60,NaT,0,0.0,4,4,4,0
293,60,NaT,2,0.0,4,4,4,2
294,60,NaT,8,2.0,4,4,4,10
295,60,NaT,37,6.0,4,4,4,43
296,60,NaT,49,13.0,4,4,4,62
297,60,NaT,52,11.0,4,4,4,63


In [11]:
# Reload the original timestamp strings
velo_original = pd.read_csv(
    velo_path,
    dtype={"stunde": "string"}
)

# Inspect the original timestamps
display(velo_original[["stunde"]].head(10))

# Parse timestamps with support for mixed formats
parsed_timestamps = pd.to_datetime(
    velo_original["stunde"].str.strip(),
    format="mixed",
    errors="coerce"
)

# Identify timestamps that could not be parsed
invalid_mask = parsed_timestamps.isna()

print("Original missing values:",
      velo_original["stunde"].isna().sum())

print("Invalid timestamps after parsing:",
      invalid_mask.sum())

display(
    velo_original.loc[
        invalid_mask,
        ["FK_STANDORT", "stunde"]
    ].head(10)
)

,stunde
0,01/01/2023 00:00
1,01/01/2023 01:00
2,01/01/2023 02:00
3,01/01/2023 03:00
4,01/01/2023 04:00
5,01/01/2023 05:00
6,01/01/2023 06:00
7,01/01/2023 07:00
8,01/01/2023 08:00
9,01/01/2023 09:00


Original missing values: 0
Invalid timestamps after parsing: 0


,FK_STANDORT,stunde


In [12]:
# Stop if invalid timestamps remain
if invalid_mask.any():
    raise ValueError(
        "Some timestamps are still invalid. "
        "Inspect them before proceeding."
    )

# Use the correctly parsed timestamps
velo = velo_original.copy()
velo["stunde"] = parsed_timestamps

# Validate bicycle merge keys
print("Missing timestamps:", velo["stunde"].isna().sum())

print(
    "Duplicate station-hour combinations:",
    velo.duplicated(
        subset=["FK_STANDORT", "stunde"]
    ).sum()
)

print("Total bicycle rows:", len(velo))

Missing timestamps: 0
Duplicate station-hour combinations: 0
Total bicycle rows: 587954


In [13]:
# Validate merge keys
assert weather_hourly["stunde"].is_unique
assert not velo["stunde"].isna().any()
assert not velo.duplicated(
    subset=["FK_STANDORT", "stunde"]
).any()

# Merge bicycle and weather data
integrated = pd.merge(
    velo,
    weather_hourly,
    on="stunde",
    how="left",
    validate="many_to_one",
    indicator=True
)

# Evaluate merge results
print("Bicycle rows before merge:", len(velo))
print("Rows after merge:", len(integrated))

print("\nMerge results:")
print(integrated["_merge"].value_counts())

# Calculate match rate
match_rate = (
    integrated["_merge"].eq("both").mean() * 100
)

print(f"\nMatch rate: {match_rate:.2f}%")

display(integrated.head())

Bicycle rows before merge: 587954
Rows after merge: 587954

Merge results:
_merge
both          587933
left_only         21
right_only         0
Name: count, dtype: int64

Match rate: 100.00%


,FK_STANDORT,stunde,velo_in,velo_out,anzahl_intervalle,messungen_in,messungen_out,velo_total,temperature_2m,precipitation,relative_humidity_2m,wind_speed_10m,cloud_cover,is_rainy,_merge
0,60,2023-01-01 00:00:00,0,1.0,4,4,4,1,NaN,NaN,NaN,NaN,NaN,NaN,left_only
1,60,2023-01-01 01:00:00,0,2.0,4,4,4,2,8.1,0.0,84.0,5.1,74.0,False,both
2,60,2023-01-01 02:00:00,0,3.0,4,4,4,3,8.2,0.0,83.0,4.6,48.0,False,both
3,60,2023-01-01 03:00:00,2,2.0,4,4,4,4,8.4,0.0,85.0,3.6,48.0,False,both
4,60,2023-01-01 04:00:00,0,0.0,4,4,4,0,8.0,0.0,86.0,2.9,61.0,False,both


## Merge Validation

The integration results are evaluated to identify
bicycle records without matching weather observations.

Unmatched timestamps and affected counting stations
are inspected before saving the integrated dataset.

In [14]:
# Select bicycle records without matching weather data
unmatched = integrated[
    integrated["_merge"] == "left_only"
].copy()

print("Unmatched records:", len(unmatched))

# Display unmatched records
display(
    unmatched[
        ["FK_STANDORT", "stunde", "velo_total"]
    ].head(30)
)

# Count unmatched records by timestamp
print("\nUnmatched timestamps:")

display(
    unmatched["stunde"]
    .value_counts()
    .sort_index()
)

# Count affected stations
print(
    "Affected counting stations:",
    unmatched["FK_STANDORT"].nunique()
)

# Show weather time coverage
print(
    "First weather timestamp:",
    weather_hourly["stunde"].min()
)

print(
    "Last weather timestamp:",
    weather_hourly["stunde"].max()
)

Unmatched records: 21


,FK_STANDORT,stunde,velo_total
0,60,2023-01-01,1
1921,732,2023-01-01,9
28198,1037,2023-01-01,29
54475,2977,2023-01-01,9
54787,2978,2023-01-01,1
56660,2982,2023-01-01,1
68515,2986,2023-01-01,4
88696,2989,2023-01-01,30
114973,2991,2023-01-01,64
138035,2993,2023-01-01,13



Unmatched timestamps:


stunde
2023-01-01    21
Name: count, dtype: int64

Affected counting stations: 21
First weather timestamp: 2023-01-01 01:00:00
Last weather timestamp: 2026-01-01 00:00:00


## Saving the Integrated Dataset

The integrated bicycle and weather dataset is saved
as a CSV file.

Unmatched bicycle observations are retained to
ensure transparency and prevent unnecessary data loss.

In [15]:
# Define output file
output_path = (
    processed_dir / "velo_weather_2023_2025.csv"
)

# Keep all bicycle records
integrated_final = integrated.copy()

# Remove the temporary merge indicator
integrated_final = integrated_final.drop(
    columns=["_merge"]
)

# Save integrated dataset
integrated_final.to_csv(
    output_path,
    index=False
)

# Validate saved dataset
print("File saved:", output_path)
print("Total rows:", len(integrated_final))
print("Total columns:", len(integrated_final.columns))

display(integrated_final.head())

File saved: c:\Projects\velo-wetter-zuerich-HS26\data\processed\velo_weather_2023_2025.csv
Total rows: 587954
Total columns: 14


,FK_STANDORT,stunde,velo_in,velo_out,anzahl_intervalle,messungen_in,messungen_out,velo_total,temperature_2m,precipitation,relative_humidity_2m,wind_speed_10m,cloud_cover,is_rainy
0,60,2023-01-01 00:00:00,0,1.0,4,4,4,1,NaN,NaN,NaN,NaN,NaN,NaN
1,60,2023-01-01 01:00:00,0,2.0,4,4,4,2,8.1,0.0,84.0,5.1,74.0,False
2,60,2023-01-01 02:00:00,0,3.0,4,4,4,3,8.2,0.0,83.0,4.6,48.0,False
3,60,2023-01-01 03:00:00,2,2.0,4,4,4,4,8.4,0.0,85.0,3.6,48.0,False
4,60,2023-01-01 04:00:00,0,0.0,4,4,4,0,8.0,0.0,86.0,2.9,61.0,False


In [16]:
# Inspect bicycle records without matching weather data
unmatched = integrated[
    integrated["_merge"] == "left_only"
].copy()

# Count unmatched records
print("Unmatched records:", len(unmatched))

# Check affected timestamps
print("\nUnmatched timestamps:")
display(
    unmatched.groupby("stunde")
    .size()
    .reset_index(name="count")
)

# Check affected counting stations
print("\nAffected stations:")
display(
    unmatched[
        ["FK_STANDORT", "stunde", "velo_total"]
    ]
)

Unmatched records: 21

Unmatched timestamps:


,stunde,count
0,2023-01-01,21



Affected stations:


,FK_STANDORT,stunde,velo_total
0,60,2023-01-01,1
1921,732,2023-01-01,9
28198,1037,2023-01-01,29
54475,2977,2023-01-01,9
54787,2978,2023-01-01,1
56660,2982,2023-01-01,1
68515,2986,2023-01-01,4
88696,2989,2023-01-01,30
114973,2991,2023-01-01,64
138035,2993,2023-01-01,13


In [ ]:
## Integration Results

The bicycle traffic and weather datasets were
successfully integrated using hourly timestamps.

### Results
- Bicycle records before merging: 587,954
- Records after merging: 587,954
- Successfully matched records: 587,933
- Unmatched bicycle records: 21
- Matching rate: approximately 99.9964%

A left join was used to preserve all bicycle traffic
observations. Weather measurements were added based
on the corresponding hourly timestamp.

Unmatched observations were retained in the
integrated dataset to avoid unnecessary data loss.

## Final Integration Results

The bicycle traffic data and weather data from Zurich were successfully integrated for the years 2023–2025.

### Integration Summary

- Bicycle records before merging: 587,954
- Records after merging: 587,954
- Successfully matched records: 587,933
- Unmatched records: 21
- Matching rate: 99.9964%
- Final dataset: 14 columns

### Missing Weather Observations

All 21 unmatched records occur on January 1, 2023, at 00:00 local time.

The weather dataset begins at 00:00 UTC, corresponding to 01:00 in Zurich. Therefore, no weather observation is available for the first local hour of the analysis period.

The unmatched bicycle observations were retained in the integrated dataset to avoid unnecessary data loss.

### Conclusion

The integration was successful, preserving all bicycle traffic records without increasing the number of observations.

The resulting dataset provides a foundation for exploratory data analysis, statistical analysis, and regression modeling.

In [17]:
# Final validation of the integrated dataset
print("Final dataset shape:", integrated_final.shape)

print(
    "Duplicate station-hour combinations:",
    integrated_final.duplicated(
        subset=["FK_STANDORT", "stunde"]
    ).sum()
)

print(
    "Missing weather observations:",
    integrated_final["temperature_2m"].isna().sum()
)

print(
    "Missing bicycle counts:",
    integrated_final["velo_total"].isna().sum()
)

# Check the analysis period
print("First timestamp:", integrated_final["stunde"].min())
print("Last timestamp:", integrated_final["stunde"].max())

# Verify the dataset was saved
print("File exists:", output_path.exists())

Final dataset shape: (587954, 14)
Duplicate station-hour combinations: 0
Missing weather observations: 21
Missing bicycle counts: 0
First timestamp: 2023-01-01 00:00:00
Last timestamp: 2025-12-31 23:00:00
File exists: True
